# Proyecto Final — Airbnb: Madrid, Mallorca, Valencia, Sevilla, Milán
## Notebook 2: Análisis Descriptivo

**Dataset:** `listings_final.csv` — 66.411 filas × 28 columnas  
**Objetivo:** Comprender la estructura del dataset, sus distribuciones y estadísticas básicas antes del análisis estadístico.

> **Nota sobre Milán:** Milán es la única ciudad no española del dataset. Los datos de turismo (Eurostat (tabla tour_occ_nin3), que incluye la región Milano (fuente ISTAT)) están disponibles para todas las ciudades. En los análisis de contexto turístico se separa España de Milán donde la comparativa lo requiere.

**Fuentes de datos:** Inside Airbnb (anuncios Airbnb) + Eurostat `tour_occ_nin3` (pernoctaciones turísticas)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)

SPAIN = ['Madrid', 'Mallorca', 'Valencia', 'Sevilla']
COUNTRY = {'Madrid':'España','Mallorca':'España','Valencia':'España','Sevilla':'España','Milan':'Italia'}

df = pd.read_csv('../data/processed/listings_final.csv', low_memory=False)
df['country'] = df['city'].map(PAIS)
df_spain = df[df['city'].isin(SPAIN)].copy()

print(f'Dataset completo: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Solo España:      {df_spain.shape[0]:,} filas')

## 1. Vista general

In [ ]:
df.head()

In [ ]:
df.info()

## 2. Tamaño muestral por país y ciudad

In [ ]:
# Tabla inicial: país + ciudad + nº anuncios + % del total
tabla_ciudad = (
    df.groupby(['country','city'])
    .size()
    .reset_index(name='n_anuncios')
    .assign(pct_total=lambda x: (x['n_anuncios'] / len(df) * 100).round(1))
    .sort_values('n_anuncios', ascending=False)
    .reset_index(drop=True)
)
print('Distribución por país y ciudad:')
tabla_ciudad

> **Conclusión:** Milán (Italia) y Madrid (España) concentran el 58% del dataset. Las ciudades españolas suman ~46.000 anuncios. Mallorca, a pesar de ser uno de los destinos turísticos más visitados de España, tiene menor número de anuncios activos, reflejo de su marcada estacionalidad.

## 3. Calidad del dato

In [ ]:
nulos = df.isna().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
con_nulos = pd.DataFrame({'Nulos': nulos, '%': nulos_pct})[nulos > 0]
print('Columnas con nulos:')
print(con_nulos if len(con_nulos) > 0 else '  Ninguna')
print(f'\nDuplicados: {df.duplicated().sum()}')

In [ ]:
print('Clasificación de nulos:')
print()
print('  Esperables (no afectan al análisis principal):')
print('    host_antiguedad_años: 111 registros (0.17%)')
print('    → Hosts sin fecha de alta en Airbnb. No afecta a precio, valoración')
print('      ni a ninguna de las hipótesis principales del análisis.')
print()
print('  Problemáticos (tratados en limpieza):')
print(f'    price:              {df["price"].isna().sum()} nulos ✅ (eliminados en limpieza)')
print(f'    review_scores_*:    0 nulos ✅ (imputados con mediana global)')
print(f'    pernoctaciones_*:   {df["pernoctaciones_total"].isna().sum()} nulos ✅ (disponible para las 5 ciudades)')

> **Conclusión:** El dataset está en excelente estado. El único nulo remanente (`host_antiguedad_años`, 0.17%) no afecta a ninguna de las hipótesis principales del análisis y se mantiene como `NaN` por ser ausencia de información legítima.
>
> El dataset presenta una **buena calidad general**: ninguna variable clave (precio, tipo de habitación, coordenadas, disponibilidad) tiene valores nulos tras la limpieza. Este nivel de calidad permite aplicar directamente los análisis estadísticos sin riesgo de sesgos por datos faltantes.
>
> **El dataset presenta alta calidad general:** ninguna variable clave como el precio, el tipo de habitación, las coordenadas o la disponibilidad contiene valores nulos tras el proceso de limpieza.

## 4. Variables numéricas — estadísticas generales

> Las variables de turismo (`pernoctaciones_total`, `tourism_pressure`) se analizan en el bloque 6 para mayor claridad.

In [ ]:
cols_num = [
    'price','price_per_person','accommodates','bathrooms','bedrooms','beds',
    'minimum_nights','availability_365','number_of_reviews',
    'review_scores_rating','review_scores_location',
    'reviews_per_month','calculated_host_listings_count','host_antiguedad_años'
]
df[cols_num].describe().round(2)

In [ ]:
print('Precio (€/noche) por ciudad:')
df.groupby('city')['price'].agg(
    Media='mean', Mediana='median', Desv_Std='std',
    P25=lambda x: x.quantile(0.25),
    P75=lambda x: x.quantile(0.75), Maximo='max'
).round(2).sort_values('Mediana', ascending=False)

> **Conclusión:** Mallorca lidera en precio mediano (239€), muy por encima del resto. La diferencia entre media (315€) y mediana (239€) en Mallorca indica un segmento de lujo que eleva la media. Valencia es la ciudad más asequible (100€ de mediana). Madrid, Milán y Sevilla presentan medianas similares (109–124€), reflejando mercados urbanos más competitivos.

In [ ]:
print('Precio por persona (€/noche) por ciudad — mejor indicador de coste real:')
df.groupby('city')['price_per_person'].agg(
    Media='mean', Mediana='median', Desv_Std='std'
).round(2).sort_values('Mediana', ascending=False)

> **Conclusión:** Al normalizar por persona, Mallorca sigue liderando pero la brecha se reduce respecto al precio bruto. Madrid y Milán tienen un precio por persona similar, lo que sugiere que Madrid compensa su mayor precio medio con alojamientos de mayor capacidad.

In [ ]:
print('Disponibilidad media (días/año) por ciudad:')
df.groupby('city')['availability_365'].agg(
    Media='mean', Mediana='median'
).round(1).sort_values('Media', ascending=False)

> **Conclusión:** Mallorca tiene la menor disponibilidad media, coherente con su estacionalidad estival. Milán y Valencia presentan los valores más altos (>200 días), indicando un mercado activo durante todo el año.

In [ ]:
print('Reseñas por ciudad:')
df.groupby('city')['number_of_reviews'].agg(
    Media='mean', Mediana='median', Maximo='max'
).round(1).sort_values('Media', ascending=False)

## 5. Variables categóricas

In [ ]:
print('Tipos de alojamiento (global):')
pd.DataFrame({
    'Nº anuncios': df['room_type'].value_counts(),
    '% del total': df['room_type'].value_counts(normalize=True).mul(100).round(1)
})

In [ ]:
print('Distribución de tipos por ciudad (%):')
room_ciudad = df.groupby(['city','room_type']).size().unstack(fill_value=0)
room_ciudad.div(room_ciudad.sum(axis=1), axis=0).mul(100).round(1)

In [ ]:
print('Precio mediano (€/noche) por ciudad y tipo de alojamiento:')
df.groupby(['city','room_type'])['price'].median().unstack().round(1)

> **Conclusión:** `Entire home/apt` domina en todas las ciudades (>79%). Mallorca presenta precios superiores en todos los tipos. En Sevilla, los `Hotel room` en Airbnb (232€) superan al `Entire home/apt` (131€), lo que indica que los hoteles sevillanos que usan Airbnb compiten en el segmento premium.

In [ ]:
bool_df = pd.DataFrame({
    'Global (%)': [
        df['host_is_superhost'].mean() * 100,
        df['high_availability'].mean() * 100,
        df['is_popular'].mean() * 100,
        df['host_profesional'].mean() * 100,
        df['instant_bookable'].mean() * 100,
    ]
}, index=['Superhost','Alta disponibilidad (>180d)','Anuncio popular','Host profesional (>3 anuncios)','Reserva instantánea'])
bool_df.round(1)

In [ ]:
print('% Superhost por ciudad:')
df.groupby('city')['host_is_superhost'].mean().mul(100).round(1).sort_values(ascending=False)

In [ ]:
print('Distribución de categoría de precio:')
pd.DataFrame({
    'Nº anuncios': df['precio_categoria'].value_counts().sort_index(),
    '% del total': df['precio_categoria'].value_counts(normalize=True).sort_index().mul(100).round(1)
})

> **Conclusión:** Solo el 29% de los hosts tienen el badge de Superhost. La distribución de categorías de precio es uniforme por diseño (cuartiles globales), pero muestra que la mitad del mercado se sitúa por debajo de 130€/noche.

## 6. Datos de turismo (Eurostat)

In [ ]:
print('Turismo y Airbnb — todas las ciudades:')
df.groupby('city').agg(
    pais=('country','first'),
    n_anuncios=('price','count'),
    precio_medio=('price','mean'),
    precio_mediano=('price','median'),
    pernoctaciones_total=('pernoctaciones_total','first'),
    tourism_pressure=('tourism_pressure','first')
).round(2).sort_values('pernoctaciones_total', ascending=False)

In [ ]:
print('Turismo y Airbnb — solo ciudades españolas (comparativa homogénea):')
df_spain.groupby('city').agg(
    n_anuncios=('price','count'),
    precio_mediano=('price','median'),
    pernoctaciones_total=('pernoctaciones_total','first'),
    tourism_pressure=('tourism_pressure','first')
).round(2).sort_values('pernoctaciones_total', ascending=False)

> **Conclusión:** Mallorca lidera en pernoctaciones totales (55M) y presión turística (1.0) en España. Sin embargo, Madrid, con 32M de pernoctaciones y el mayor volumen de anuncios Airbnb, presenta un precio mediano inferior al de Mallorca. La relación entre turismo y precio no es lineal: la abundante oferta en Madrid modera los precios a pesar del volumen turístico.

## 7. Resumen ejecutivo

| Indicador | Valor |
|---|---|
| Total anuncios | 66.411 |
| Precio medio global | 178.7 €/noche |
| Precio mediano global | 130 €/noche |
| Ciudad más cara (mediana) | **Mallorca** — 239€/noche |
| Ciudad más asequible (mediana) | **Valencia** — 100€/noche |
| Tipo de alojamiento dominante | **Entire home/apt** — 83.2% |
| Mayor disponibilidad media | **Milán y Valencia** — >200 días/año |
| % Superhosts global | **29.1%** |
| Mayor presión turística (España) | **Mallorca** — 55M pernoctaciones |
| Nulos problemáticos | **0** |

**Hallazgos clave:**
- El precio presenta **fuerte asimetría a la derecha**: la media (178€) supera ampliamente la mediana (130€), revelando un segmento premium que eleva la distribución.
- **Mallorca** es un mercado estructuralmente diferente: precios medianos un 139% superiores a Valencia, estacionalidad pronunciada y mayor presión turística.
- El tipo `Entire home/apt` domina en todas las ciudades, con diferencias de precio notables según ciudad y barrio.
- **Mayor presión turística no implica mayor precio Airbnb**: la oferta de alojamiento, el tipo de turismo y la regulación local modulan esta relación (Madrid vs Mallorca).
- El **precio por persona** es una métrica más equitativa que el precio bruto para comparar entre ciudades con capacidades medias distintas.

---

**Conclusión del análisis descriptivo:** Se observan diferencias claras y consistentes entre ciudades. Mallorca y Milán destacan como los mercados con precios más elevados, mientras que Valencia es la opción más asequible del dataset. El tipo de alojamiento dominante es `Entire home/apt` en todas las ciudades, y la presión turística no explica por sí sola el nivel de precios — la oferta disponible y el tipo de turismo son factores moduladores clave.